In [ ]:
import json
from pathlib import Path
from notebook_utils import (
    get_model_schedule, stage_label,
    plot_interf_batch_times, plot_interf_throughput,
    print_interf_throughput_impact, plot_interf_optimizer_state,
    plot_interf_boxplot,
)
import numpy as np

%matplotlib inline
import matplotlib.pyplot as plt
plt.rcParams['figure.dpi'] = 120

## Configuration

In [ ]:
INTERFERENCE_DIR = Path("interference")

# Select which interference run to load (filename without .json, or None for latest)
SELECTED_RUN = "interf6"

# Optional: a non-interference baseline run from data/runs/ for comparison
BASELINE_RUN = "run41"
RUNS_DIR = Path("runs")

# Display options
SHOW_REBALANCE = False
SHOW_OPTIMUM = True
SHOW_INTERFERENCE_REGIONS = True  # shade background by interference step

In [ ]:
# Load interference run
if SELECTED_RUN:
    interf_path = INTERFERENCE_DIR / f"{SELECTED_RUN}.json"
else:
    interf_files = sorted(INTERFERENCE_DIR.glob("*.json"))
    interf_path = interf_files[-1] if interf_files else None

with open(interf_path) as f:
    interf_data = json.load(f)

print(f"Loaded: {interf_path.stem}")

# Load baseline if specified
baseline_data = None
if BASELINE_RUN:
    baseline_path = RUNS_DIR / f"{BASELINE_RUN}.json"
    if baseline_path.exists():
        with open(baseline_path) as f:
            baseline_data = json.load(f)
        print(f"Baseline: {BASELINE_RUN}")
    else:
        print(f"Baseline not found: {baseline_path}")

## Run Summary

In [ ]:
meta = interf_data.get("meta", {})
print(f"Schedule: {meta.get('schedule')}")
print(f"Optimizer: {meta.get('optimizer')}, kwargs: {meta.get('optimizer_kwargs')}")
print()

for model, result in interf_data["results"].items():
    steps, dur = get_model_schedule(interf_data, model)
    print(f"--- {model} ({len(steps)} steps x {dur}s) ---")
    for i, step in enumerate(steps):
        print(f"  Step {i}: {stage_label(step) if step else 'idle'}")

    batches = result.get("batches", [])
    timed = [b for b in batches if "timing" in b]
    rps = result.get("requests_per_second", 0)
    rebalances = sum(1 for b in batches if b.get("rebalance", {}).get("did_rebalance", False))
    at_optimum = sum(1 for b in batches if b.get("rebalance", {}).get("at_optimum", False))
    if timed:
        times = [b["timing"]["end"] - b["timing"]["start"] for b in timed]
        wall = timed[-1]["timing"]["end"] - timed[0]["timing"]["start"]
        print(f"  rps={rps:.2f}, batches={len(timed)}, rebalances={rebalances}, at_optimum={at_optimum}")
        print(f"  forward: avg={np.mean(times):.3f}s, min={min(times):.3f}s, max={max(times):.3f}s, wall={wall:.0f}s")
    print()

if baseline_data:
    print(f"--- Baseline ({BASELINE_RUN}) ---")
    for model, result in baseline_data["results"].items():
        print(f"  {model}: rps={result.get('requests_per_second', 0):.2f}")

## Batch Times with Interference Regions

In [ ]:
plot_interf_batch_times(interf_data, baseline_data, baseline_name=BASELINE_RUN,
                       show_interference_regions=SHOW_INTERFERENCE_REGIONS,
                       show_optimum=SHOW_OPTIMUM)

## Throughput Comparison (RPS)

In [ ]:
plot_interf_throughput(interf_data, baseline_data, baseline_name=BASELINE_RUN)

## Throughput Impact (%)

In [ ]:
print_interf_throughput_impact(interf_data, baseline_data, baseline_name=BASELINE_RUN)

## Optimizer State Under Interference

In [ ]:
plot_interf_optimizer_state(interf_data,
                           show_interference_regions=SHOW_INTERFERENCE_REGIONS,
                           show_optimum=SHOW_OPTIMUM)

## Forward Time Distribution per Interference Step

In [ ]:
plot_interf_boxplot(interf_data)